In [19]:
import requests

In [20]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
relative_path_from_repo_name = 'archive/refs/heads'
repo_branch_name = 'main'

# We are actually saving the repo but having it in memory
zip_url = f'https://github.com/{repo_owner}/{repo_name}/{relative_path_from_repo_name}/{repo_branch_name}.zip'
# Download the repository ZIP and keep it in memory (RAM not ROM). so if kernel is lost download data is lost 
zip_response = requests.get(zip_url)

In [21]:
len(zip_response.content)

17544837

In [22]:
import io
import zipfile

zip_archived = zipfile.ZipFile(io.BytesIO(zip_response.content))
zip_archived

<zipfile.ZipFile file=<_io.BytesIO object at 0x7d928725c0e0> mode='r'>

In [23]:
filenames = zip_archived.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [26]:
file_to_open_path = 'docs-main/docs/platform/alerts.mdx'
md_file = zip_archived.open(file_to_open_path)
md_content = md_file.read().decode('utf8')
print(md_content)

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.
</Check>

![](/images/alerts.png)

To enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:

* A notification channel.

* An alert condition.

## Notification channels

You can choose between the following options:

* **Email**. Add email addresses to send alerts to.

* **Slack**. Add a Slack webhook.

* **Discord**. Add a Discord webhook.

## Alert conditions

### Failed tests

If you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.

<Tip>
  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if th

In [ ]:
#!uv add python-frontmatter

Resolved 113 packages in 2.10s                                       
Prepared 3 packages in 49ms                                                  python-frontmatter   ------------------------------ 10.31 KiB/10.31 KiB         python-frontmatter   ------------------------------ 10.31 KiB/10.31 KiB         
Uninstalled 4 packages in 504ms
░░░░░░░░░░░░░░░░░░░░ [0/5] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 5 packages in 5.37s                               
 ~ aiphase==0.1.0 (from file:///workspaces/aiphase)
 - fqdn==1.6.0
 + fqdn==1.5.1
 - openai==3.20.0
 + openai==3.19.2
 - platformdirs==4.12.1
 + platformdirs==4.11.12
 + python-frontmatter==1.3

In [29]:
import frontmatter

post = frontmatter.loads(md_content)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [32]:
print(post.metadata)

{'title': 'Alerts', 'description': 'How to set up alerts.'}


In [35]:
file_to_open_path = 'docs-main/docs/platform/alerts.mdx'
print(file_to_open_path.split('/'))
print(file_to_open_path.split('/', maxsplit=2))
print(file_to_open_path.split('/', maxsplit=1))

['docs-main', 'docs', 'platform', 'alerts.mdx']
['docs-main', 'docs', 'platform/alerts.mdx']
['docs-main', 'docs/platform/alerts.mdx']


In [39]:
needed_file_to_open_path1 = file_to_open_path.split('/', maxsplit=1)[-1]  #get the last one from ['docs-main', 'docs/platform/alerts.mdx']  which is  docs/platform/alerts.mdx
print(needed_file_to_open_path1)

_, needed_file_to_open_path2 = file_to_open_path.split('/', maxsplit=1)  #['docs-main', 'docs/platform/alerts.mdx'] becomes _ and needed_file_to_open_path2 becomes 'docs/platform/alerts.mdx'
print(needed_file_to_open_path2)

docs/platform/alerts.mdx
docs/platform/alerts.mdx


In [42]:
#sowe need to do it for all the files in zip_archived.namelist() to look in the format below
#remember post = frontmatter.loads(md_content)
# structure of data we want for each file in zip_archived.namelist()

one_file_details = {
    'content': post.content[:50],
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': needed_file_to_open_path1
}

print(one_file_details)

{'content': '<Check>\n  Built-in alerting is a Pro feature avail', 'title': 'Alerts', 'description': 'How to set up alerts.', 'filename': 'docs/platform/alerts.mdx'}


In [43]:
def read_github_repository(repo_owner, repo_name, repo_branch_name='main'):    
    # #The code to iterate all the file list when I get the url of the github zip
    # repo_owner = 'evidentlyai'
    # repo_name = 'docs'
    # repo_branch_name = 'main'

    # We are actually saving the repo but having it in memory
    zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{repo_branch_name}.zip'
    # Download the repository ZIP and keep it in memory (RAM not ROM). If the kernel restarts, these in-memory Python objects are lost.
    zip_response = requests.get(zip_url)
    zip_response.raise_for_status()

    git_zip_files_data = []

    with zipfile.ZipFile(io.BytesIO(zip_response.content)) as contained_in_zip:

        for file_path in contained_in_zip.namelist():
            # Only ingest text/documentation files
            if file_path.endswith(('.md', '.mdx', '.txt')):

                with contained_in_zip.open(file_path) as file:
                    file_content= file.read().decode('utf-8', errors='replace')
                    
                    file_ref= frontmatter.loads(file_content)
                    g_zip_file_data= {
                        'content': file_ref.content,
                        'title': file_ref.metadata.get('title'),
                        'description': file_ref.metadata.get('description'),
                        'filename': file_path.split('/', 1)[-1]
                    }
                    git_zip_files_data.append(g_zip_file_data)
            

    return git_zip_files_data



In [44]:
repo_files = read_github_repository(
    repo_owner='evidentlyai',
    repo_name='docs',
    repo_branch_name='main'
)
len(repo_files)

95

In [53]:
#repo_files[2]

first_file = repo_files[6]

print("TITLE:")
print(first_file['title'])

print("\nFILENAME:")
print(first_file['filename'])

print("\nCONTENT:")
print(first_file['content'][:100])

TITLE:
Descriptors

FILENAME:
docs/library/descriptors.mdx

CONTENT:
To evaluate text data, like LLM inputs and outputs, you create **Descriptors**. This is a universal 


In [ ]:
#!uv add gitsource
#uv add git+https://github.com/alexeygrigorev/gitsource.git  for another repo not saved in PyPI

Resolved 114 packages in 1.18s                                       
Prepared 2 packages in 70ms                                                  gitsource            ------------------------------ 8.77 KiB/8.77 KiB           
Uninstalled 1 package in 29ms
░░░░░░░░░░░░░░░░░░░░ [0/2] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 14msfile:///workspaces/aiphase)     
 ~ aiphase==0.1.0 (from file:///workspaces/aiphase)
 + gitsource==0.0.5


In [55]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx", "txt"},
)

files = reader.read()

print(f'Loaded {len(files)} files')

Loaded 95 files


In [57]:
files[6].filename

'docs/library/descriptors.mdx'

In [ ]:
# iterate and parse all files. parse using gitsource functions still
documents = [f.parse() for f in files]      #[result_you_want for item in collection]  ie For every f in files, run f.parse(), and put the result into a new list called documents
#same as this

# documents = []
# for f in files:
#     parsed_document = f.parse()
#     documents.append(parsed_document)



len(documents)

95

In [59]:
documents[6]

{'title': 'Descriptors',
 'description': 'How to run evaluations for text data.',
 'content': 'To evaluate text data, like LLM inputs and outputs, you create **Descriptors**. This is a universal interface for all evals - from text statistics to LLM judges.\n\nEach descriptor computes a score or label per row of your dataset. You can combine multiple descriptors and set optional pass/fail conditions. You can use built-in descriptors or create custom ones using LLM prompts or Python.\n\nFor a general introduction, check [Core Concepts](/docs/library/overview). You can also refer to the [LLM quickstart](quickstart_llm) for a minimal example.\n\n## Basic flow\n\n<Accordion title="Generate toy data" defaultOpen={false}>\n  Use this code snippet to create sample data for testing:\n\n  ```python\n  import pandas as pd\n  \n  data = [\n      ["What is the chemical symbol for gold?", "The chemical symbol for gold is Au."],\n      ["What is the capital of Japan?", "The capital of Japan is Tokyo.